# PPO Metric Reosurce Managment

In [1]:
import sys
import numpy as np

import stable_baselines3 as sb3
print("sb3", sb3.__version__)
print(sys.executable)
print("numpy", np.__version__)

import torch
print("torch", torch.__version__)
print("torch config",   torch.__config__.show())

sb3 2.8.0
/Users/guyarieli/PycharmProjects/resource_managment/.venv/bin/python
numpy 2.2.6
torch 2.11.0
torch config PyTorch built with:
  - GCC 4.2
  - C++ Version: 201703
  - clang 15.0.0
  - OpenMP 201811
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: DEFAULT
  - Build settings: BLAS_INFO=accelerate, BUILD_TYPE=Release, COMMIT_SHA=70d99e998b4955e0049d13a98d77ae1b14db1f45, CXX_COMPILER=/usr/bin/c++, CXX_FLAGS= -fvisibility-inlines-hidden -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO -DLIBKINETO_NOCUPTI -DLIBKINETO_NOROCTRACER -DLIBKINETO_NOXPUPTI=ON -DUSE_PYTORCH_QNNPACK -DAT_BUILD_ARM_VEC256_WITH_SLEEF -DUSE_XNNPACK -DUSE_PYTORCH_METAL_EXPORT -DSYMBOLICATE_MOBILE_DEBUG_HANDLE -DUSE_COREML_DELEGATE -O2 -fPIC -DC10_NODEPRECATED -Wall -Wextra -Werror=return-type -Werror=non-virtual-dtor -Werror=braced-scalar-init -Werror=range-loop-construct -Werror=bool-operation -Wnarrowing -Wno-missing-field-initializers -Wno-unknown-pragmas -Wno-unused-par

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

from environment.wrappers.time_limit_penalty import TimeLimitPenaltyWrapper
from environment.wrappers.action_flattener import FlattenActionWrapper
from environment.wrappers.zero_by_status import ZeroJobUsageByTheirStatus
from environment.core.jobs import JobStatus
from experiments.utils.wandb_custome_metric import CustomMetricsCallback
from environment.envs import *

objc[89355]: Class SDL_RumbleMotor is implemented in both /Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x13fa3cd40) and /Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x1564809c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[89355]: Class SDL_RumbleContext is implemented in both /Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x13fa3cd90) and /Users/guyarieli/PycharmProjects/resource_managment/.venv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x156480a18). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[89355]: Class SDLApplication is implemented in both /Users/guyarieli/PycharmProjects/resource_manag

In [3]:
from stable_baselines3.common.callbacks import CallbackList
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback
import logging

logging.basicConfig(level=logging.ERROR)

In [4]:
N_JOBS = 10
N_MACHINES = 1
N_RESOURCES = 2
N_TICKS = 5
OFFLINE = True

### Simple NN
Create simple nn where there is no special valeus

In [5]:
total_timesteps = 100_000

policy_kwargs = dict(
    net_arch=[64,32,64]
)

config = {
    "policy_type": "MultiInputPolicy",
    "total_timesteps": total_timesteps,
    "env_name": "resource-management-online",
}

In [6]:
run = wandb.init(
    project="cluster-scheduling",
    config=config,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/guyarieli/.netrc.
wandb: Currently logged in as: guyarieli17 (dev0guy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
from environment.wrappers.metric_reward import MetricRewardWrapper
from environment.wrappers.invalid_metric_action_masker import InvalidMetricActionMasker
from sb3_contrib.common.wrappers import ActionMasker

def mask_fn(env) -> np.ndarray:
    return env.valid_action_mask()
    

def make_env(
    n_jobs: int,
    n_machines: int,
    n_resources: int,
    n_ticks: int,
    max_episode_steps: int = 300,
    offline: bool = True,
):
    print(f"{n_jobs=}, {n_machines=}, {n_resources=}, {n_ticks=}, {max_episode_steps=}")
    env = gym.make(
        config["env_name"],
        render_mode="rgb_array",
        n_jobs=n_jobs,
        n_machines=n_machines,
        n_resources=n_resources,
        n_ticks=n_ticks,
        offline=offline,
    )
    # env = MetricRewardWrapper(env)
    env = ZeroJobUsageByTheirStatus(env, JobStatus.Running, JobStatus.Completed, JobStatus.NotCreated)
    print(env.action_space[0])
    env = InvalidMetricActionMasker(env)
    env = ActionMasker(env, mask_fn)
    env = Monitor(env)
    return env

In [15]:
from sb3_contrib import MaskablePPO # TODO: try with MaskablePPO

env = DummyVecEnv([lambda: make_env(N_JOBS, N_MACHINES, N_RESOURCES, N_TICKS, offline=OFFLINE)])
env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda x: x % 2_000 == 0,
    video_length=200,
)

model = MaskablePPO(
    config["policy_type"],
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log=f"runs/{run.id}"
)
wandb_callback = WandbCallback(
    gradient_save_freq=1_000,
    model_save_path=f"models/{run.id}",
    verbose=2,
)
metric_callback = CustomMetricsCallback()
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=CallbackList([
        metric_callback,
        wandb_callback
    ]),
)
model.save(f"models/{run.id}/final_model")
wandb.save(f"models/{run.id}/final_model.zip")
import glob
for f in glob.glob(f"videos/{run.id}/*.mp4"):
    wandb.log({"video": wandb.Video(f, fps=30, format="mp4")})
wandb.finish()

n_jobs=10, n_machines=1, n_resources=2, n_ticks=5, max_episode_steps=300
Discrete(1)


TypeError: 'Discrete' object is not subscriptable

In [ ]:
##%% md
### Evaluate Best Model with W&B
##%%
import glob
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
from stable_baselines3.common.monitor import Monitor

MODEL_PATH = f"models/{run.id}/final_model.zip"
EPISODES = 100

eval_run = wandb.init(
    project="cluster-scheduling",
    job_type="evaluation",
    config={
        "model_path": MODEL_PATH,
        "episodes": EPISODES,
        "n_jobs": N_JOBS,
        "n_machines": N_MACHINES,
        "n_resources": N_RESOURCES,
        "n_ticks": N_TICKS,
        "offline": OFFLINE,
    }
)

# --- env ---
vec_env = DummyVecEnv([lambda: make_env(N_JOBS, N_MACHINES, N_RESOURCES, N_TICKS, offline=OFFLINE)])
vec_env = VecVideoRecorder(
    vec_env,
    f"videos/{eval_run.id}",
    record_video_trigger=lambda x: x == 0,
    video_length=10_000,
)

model = PPO.load(MODEL_PATH, env=vec_env)

# --- run episodes ---
all_rewards, all_steps = [], []

for ep in range(EPISODES):
    obs = vec_env.reset()
    total_reward, step, done = 0.0, 0, False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_env.step(action)
        total_reward += float(reward[0])
        step += 1
        done = done[0]
        if step > 300:
            break

    all_rewards.append(total_reward)
    all_steps.append(step)

    wandb.log({
        "eval/episode":      ep + 1,
        "eval/reward":       total_reward,
        "eval/steps":        step,
    })
    print(f"Episode {ep+1}: reward={total_reward:.2f}, steps={step}")

# --- summary ---
wandb.summary["eval/mean_reward"] = sum(all_rewards) / EPISODES
wandb.summary["eval/mean_steps"]  = sum(all_steps)   / EPISODES
wandb.summary["eval/total_steps"] = sum(all_steps)

print(f"\navg reward: {sum(all_rewards)/EPISODES:.2f} | avg steps: {sum(all_steps)/EPISODES:.1f}")

# --- upload videos (must close env first!) ---
vec_env.close()
for f in glob.glob(f"videos/{eval_run.id}/*.mp4"):
    wandb.log({"eval/video": wandb.Video(f, format="mp4")})

wandb.finish()

### Simple NN With Status Zeroing

In [ ]:
total_timesteps =100_000

policy_kwargs = dict(
    net_arch=[32,64,32]
)

config = {
    "policy_type": "MultiInputPolicy",
    "total_timesteps": total_timesteps,
    "env_name": "resource-management-online",
}

In [ ]:
run = wandb.init(
    project="cluster-scheduling",
    config=config,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=True,
)

In [ ]:
def make_env(
    n_jobs: int,
    n_machines: int,
    n_resources: int,
    n_ticks: int,
    max_episode_steps: int = 300,
    offline: bool = True,
):
    print(f"{n_jobs=}, {n_machines=}, {n_resources=}, {n_ticks=}, {max_episode_steps=}")
    env = gym.make(
        config["env_name"],
        render_mode="rgb_array",
        n_jobs=n_jobs,
        n_machines=n_machines,
        n_resources=n_resources,
        n_ticks=n_ticks,
        offline=offline,
    )
    env = ZeroJobUsageByTheirStatus(env, JobStatus.Running, JobStatus.Completed, JobStatus.NotCreated)
    env = Monitor(env)
    return env

In [ ]:
env = DummyVecEnv([lambda: make_env(5, 1, 2, 5, offline=False)])
env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda x: x % 2_000 == 0,
    video_length=200,
)
model = PPO(
    config["policy_type"],
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log=f"runs/{run.id}"
)
wandb_callback = WandbCallback(
    gradient_save_freq=1_000,
    model_save_path=f"models/{run.id}",
    verbose=2,
)
metric_callback = CustomMetricsCallback()
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=CallbackList([
        metric_callback,
        wandb_callback
    ]),
)
model.save(f"models/{run.id}/final_model")
wandb.save(f"models/{run.id}/final_model.zip")


In [ ]:
!pip list

In [ ]:
"""
run_model.py — Load a trained PPO model and run it in the environment.

Usage:
  python run_model.py --model_path models/<run_id>/best_model.zip
  python run_model.py --model_path models/<run_id>/best_model.zip --episodes 5
"""

import os
import random

import numpy as np

from src.experiment.common.wrappers import TimeLimitPenaltyWrapper, FlattenActionWrapper

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import argparse
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from src.experiment.common.wrappers_new.job_zero_by_status import ZeroJobsByStatusWrapper
from src.server_simulator.envs.cluster_simulator.base.extractors.reward import (
    NegativeRewardForStepWithTerminalBonus,
)
# from src.experiment.common.wrappers import FlattenActionWrapper, TimeLimitPenaltyWrapper


def make_env(n_jobs=10, n_machines=1, n_resources=2, n_ticks=10,
             max_episode_steps=100, penalty=-1e4, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    env = gym.make(
        "ClusterScheduling-metric-offline-v1",
        render_mode="rgb_array",
        n_jobs=n_jobs,
        n_machines=n_machines,
        n_resources=n_resources,
        n_ticks=n_ticks,
        reward_caculator=NegativeRewardForStepWithTerminalBonus(),
    )
    env = TimeLimitPenaltyWrapper(env, max_episode_steps=max_episode_steps, penalty=penalty)
    env = ZeroJobsByStatusWrapper(env)
    env = FlattenActionWrapper(env)
    env = Monitor(env)
    return env


def run(model_path: str, episodes: int = 3):
    print(f"Loading model from: {model_path}\n")
    env = make_env()
    model = PPO.load(model_path, env=env)

    for ep in range(episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        step = 0
        done = False

        print(f"── Episode {ep + 1} ──────────────────────────")

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            step += 1
            done = terminated or truncated
            print(f"  step {step:>3}  action={action}  reward={reward:>10.2f}  total={total_reward:>10.2f}")

        print(f"  → done in {step} steps  |  total reward: {total_reward:.2f}\n")

    env.close()


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model_path", type=str, required=True, help="Path to .zip model checkpoint")
    p.add_argument("--episodes",   type=int, default=3,    help="Number of episodes to run")
    args = p.parse_args()

    run(args.model_path, args.episodes)

In [ ]:
make_env(5, 3, 2, 10).observation_space, make_env(5, 3, 2, 10).action_space